In [1]:
%load_ext autoreload

```python

import ollama
from loguru import logger
from selenium.webdriver.common.by import By
from selenium.webdriver.remote.webelement import WebElement
```


In [3]:
from langchain import hub

pr = hub.pull("wfh/web-voyager")

/Users/arseniy/Library/Caches/pypoetry/virtualenvs/mabot-VHhL2YHK-py3.11/lib/python3.11/site-packages/langsmith/client.py:253: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


```python
%autoreload 2
from mabot.browser.browser import TraverseBrowserElements, init_driver
```


```python
driver = init_driver()
driver.get(
    "https://stackoverflow.com/questions/73365490/how-to-list-all-clickable-links-in-a-website-using-selenium-python"
)
```


```python
body = driver.find_elements(by=By.XPATH, value="//body")[0]
body
```


```python
TraverseBrowserElements(driver).dfs_displayed(
    body, callback=lambda x: print(x.tag_name, hasattr(x, "send_keys"))
)
```


```python
window_location = driver.get_window_position()
window_size = driver.get_window_size()
window_width, window_height = window_size["width"], window_size["height"]
```


```python
def ask_visual_caption(element: WebElement) -> list[str]:
    path = f"./img/{element.id}.png"
    element.screenshot(path)
    result = ollama.chat(
        model="llava:7b",
        messages=[
            {
                "role": "user",
                "content": "This is the screenshot of the element on website. What it contains?",
                "images": [path],
            }
        ],
    )
    return result["message"]["content"]
```


```python
prompt = """
Analyse website DOM element and give possible user actions with this element.
Answer in a list.

# DOM:
{dom}
"""


def ask_element_purpose(element: WebElement | dict) -> list[str]:
    result = ollama.generate("mistral", prompt.format(dom=element))["response"]
    logger.info("Element: {}", element)
    logger.info("Purpose: {}", result)
    return result
```


```python
def DFS(root: WebElement, callback):
    path_id: list[str] = []
    result = []
    stack: list[tuple[WebElement, WebElement]] = []
    stack.append((root, root))

    while stack:
        parent, current = stack.pop()
        if current.id not in path_id:
            result.append(callback(parent, current))
            path_id.append(current.id)

        elif current in path_id:
            # leaf node
            continue
        for tag in current.find_elements(by=By.XPATH, value="child::*"):
            if tag.is_displayed():
                stack.append((current, tag))

    return result
```


```python
class AggPath:

    def __init__(self):
        self.nodes = {}
        self.edges = {}
        self.llm = {}

    def __call__(self, parent: WebElement, current: WebElement):

        location = current.location
        relative_x = location["x"] - window_location["x"]
        relative_y = location["y"] - window_location["y"]
        size = current.size
        width, height = size["width"], size["height"]
        width_ratio = width / window_width
        height_ratio = height / window_height
        x_ratio = relative_x / window_width
        y_ratio = relative_y / window_height

        self.edges[parent.id] = current.id
        attrs = driver.execute_script(
            "var items = {}; for (index = 0; index < arguments[0].attributes.length; ++index) { items[arguments[0].attributes[index].name] = arguments[0].attributes[index].value }; return items;",
            current,
        )
        attrs["x_relative"] = x_ratio
        attrs["y_relative"] = y_ratio
        attrs["width_relative"] = width_ratio
        attrs["height_relative"] = height_ratio
        attrs["tag_name"] = current.tag_name
        attrs["text"] = current.text

        # self.llm[current.id] = ask_element_purpose(attrs)
        ask_visual_caption(current)
        self.nodes[current.id] = attrs

        # if parent.id in self.nodes:

        # else:
        #     self.nodes[current.id] = list(attrs.keys())
```


```python
agg = AggPath()
DFS(body[0], agg)
```


```python
agg.nodes
```


```python
attrs = driver.execute_script(
    "var items = {}; for (index = 0; index < arguments[0].attributes.length; ++index) { items[arguments[0].attributes[index].name] = arguments[0].attributes[index].value }; return items;",
    body[0],
)
```


```python
attrs
```


```python
children = body[0].find_elements(by=By.XPATH, value="child::*")
len(children)
```


```python
[x.tag_name for x in children]
```


```python
driver.find_element(By.CSS_SELECTOR, "canvas")
```


Imagine you are a robot browsing the web, just like humans. Now you need to complete a task. In each iteration, you will receive an Observation that includes a screenshot of a webpage and some texts. This screenshot will
feature Numerical Labels placed in the TOP LEFT corner of each Web Element. Carefully analyze the visual
information to identify the Numerical Label corresponding to the Web Element that requires interaction, then follow
the guidelines and choose one of the following actions:

1. Click a Web Element.
2. Delete existing content in a textbox and then type content.
3. Scroll up or down.
4. Wait 
5. Go back
7. Return to google to start over.
8. Respond with the final answer

Correspondingly, Action should STRICTLY follow the format:

- Click [Numerical_Label] 
- Type [Numerical_Label]; [Content] 
- Scroll [Numerical_Label or WINDOW]; [up or down] 
- Wait 
- GoBack
- Google
- ANSWER; [content]

Key Guidelines You MUST follow:

* Action guidelines *
1) Execute only one actio

input_variables=['bbox_descriptions', 'img', 'input'] optional_variables=['scratchpad'] input_types={'scratchpad': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChun

{'url': 'data:image/png;base64,{img}'}


In [17]:
from langchain_ollama import ChatOllama

In [ ]:
chat_llm = ChatOllama(model="llama3.1", temperature=0)

In [20]:
chat_llm

ChatOllama(model='llama3.1', temperature=0.0)

In [21]:
2**128

340282366920938463463374607431768211456

```python
path = "marked_screenshot.png"

prompt = """
Imagine you are a robot browsing the web, just like humans.
Now you need to complete a task.
In each iteration, you will receive an Observation that includes a screenshot of a webpage and some texts.
This screenshot will feature Numerical Labels placed in the TOP LEFT corner of each Web Element.
Carefully analyze the visual information to identify the Numerical Label corresponding to the Web Element that requires interaction.

What is the title?
"""
result = ollama.chat(
    model="llama3.2-vision",
    messages=[
        {
            "role": "user",
            "content": prompt,
            "images": [path],
        }
    ],
)
```


The title is "Python open html file, take screenshots".


In [3]:
x = 500 * 2.9232 + 500 * 2.9171 + 1000 * 2.9171 + 1000 * 2.8852 + 1000 * 2.9182
x

11640.650000000001